# Lábia — avaliação dos modelos de ponta pelas APIs gratuitas

Compara o modelo pequeno com dois modelos grandes, chamados pelas camadas gratuitas:

| Modelo | Por onde | Cenários |
|---|---|---|
| DeepSeek-V4.1-Flash (padrão e com prompt reforçado) | teste gratuito da NVIDIA | todos os 212, 3 rodadas |
| Gemini Flash-Lite | camada gratuita do Google AI Studio | um subconjunto, 1 rodada, por causa do limite diário |

Os termos dessas camadas permitem **teste e avaliação**, não gerar dado de treino. Por isso nada daqui entra no treino, e destas conversas publicamos as métricas, não as conversas em massa.

O cliente simulado é o mesmo das outras avaliações (Ministral-3-14B, na GPU deste notebook), para a comparação ser justa.

**As chaves vêm dos Kaggle Secrets** (`NVIDIA_API_KEY` e `GOOGLE_AI_STUDIO_API_KEY`), anexados a este notebook. Elas nunca aparecem no código nem na saída.

In [ ]:
import json, os, shutil, subprocess, sys, time, urllib.request
from pathlib import Path
LABIA_COMMIT = "c7a683c51b61aa581febee3cc31c0dff02c4756a"
SIMULATOR = "ministral-3:14b-instruct-2512-q4_K_M"
OUT = Path("/kaggle/working/avaliacao-api"); OUT.mkdir(parents=True, exist_ok=True)
CONFIGS = [
    {"model_id": "deepseek", "provider": "nvidia", "model": "deepseek-ai/deepseek-v4.1-flash", "defense": "standard", "subset": None, "runs": 3, "rpm": 35, "seed": True},
    {"model_id": "deepseek+reforco", "provider": "nvidia", "model": "deepseek-ai/deepseek-v4.1-flash", "defense": "reinforced", "subset": None, "runs": 3, "rpm": 35, "seed": True},
    {"model_id": "gemini-lite", "provider": "google", "model": "auto", "defense": "standard", "subset": 60, "runs": 1, "rpm": 10, "seed": False},
]
ENDPOINTS = {"nvidia": "https://integrate.api.nvidia.com/v1", "google": "https://generativelanguage.googleapis.com/v1beta/openai/"}
r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", f"git+https://github.com/deniscarvalho-tech/labia.git@{LABIA_COMMIT}"],
                   capture_output=True, text=True)
print((r.stdout + r.stderr)[-300:])

## 1. Chaves (Kaggle Secrets)

Sem a chave de um provedor, as configurações dele são puladas, e o notebook segue com as outras.

In [ ]:
KEYS = {}
try:
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
    for provider, name in [("nvidia", "NVIDIA_API_KEY"), ("google", "GOOGLE_AI_STUDIO_API_KEY")]:
        try:
            KEYS[provider] = secrets.get_secret(name)
            print(f"{name}: encontrada")
        except Exception:
            print(f"{name}: não anexada a este notebook (Add-ons > Secrets)")
except ImportError:
    print("fora do Kaggle: sem Secrets")
CONFIGS = [c for c in CONFIGS if c["provider"] in KEYS]
assert CONFIGS, "nenhuma chave anexada: veja docs/como-rodar.md, seção de avaliação pelas APIs"

## 2. O nome do modelo Flash-Lite

O nome exato muda com as versões e varia por conta. Em vez de fixar um nome que pode estar errado, perguntamos à própria API quais modelos a chave enxerga e ficamos com o Flash-Lite mais recente.

In [ ]:
from openai import OpenAI
for c in CONFIGS:
    if c["model"] == "auto":
        ids = sorted(m.id for m in OpenAI(base_url=ENDPOINTS["google"], api_key=KEYS["google"]).models.list().data)
        lite = [i for i in ids if "flash-lite" in i and "preview" not in i and "exp" not in i] or [i for i in ids if "flash-lite" in i]
        c["model"] = lite[-1].removeprefix("models/")
        print("Flash-Lite escolhido:", c["model"], "| disponíveis:", [i for i in ids if "flash" in i])

## 3. O simulador na GPU

In [ ]:
subprocess.run("apt-get -qq update > /dev/null 2>&1; apt-get -qq install -y zstd pciutils > /dev/null 2>&1", shell=True)
subprocess.run("curl -fsSL https://ollama.com/install.sh | sh", shell=True, capture_output=True)
env = dict(os.environ, OLLAMA_HOST="127.0.0.1:11435", OLLAMA_MODELS="/tmp/ollama-sim", OLLAMA_NUM_PARALLEL="8",
           OLLAMA_CONTEXT_LENGTH="6144", OLLAMA_FLASH_ATTENTION="1", OLLAMA_KV_CACHE_TYPE="q8_0", OLLAMA_KEEP_ALIVE="-1",
           OLLAMA_SCHED_SPREAD="1")
subprocess.Popen(["ollama", "serve"], env=env, stdout=open("/kaggle/working/ollama.log", "w"), stderr=subprocess.STDOUT)
for _ in range(60):
    try:
        urllib.request.urlopen("http://127.0.0.1:11435/api/tags", timeout=2); break
    except Exception:
        time.sleep(1)
subprocess.run(["ollama", "pull", SIMULATOR], env=env, check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
print("simulador pronto")

## 4. Rodar

O freio de ritmo (`RateLimitedClient`) mantém cada provedor abaixo do limite por minuto. O subconjunto do Gemini é fixo e estratificado: metade com táticas vistas, metade com a tática nunca vista, metade golpe e metade legítimo.

In [ ]:
import random
from collections import Counter
from labia.agents import LLMAgent, LLMCustomer
from labia.contracts.common import Defense, Tactic
from labia.llm import CachedClient, OpenAICompatClient, RateLimitedClient
from labia.runner import Job, load_records, run_batch, run_seed
from labia.scenarios.generator import generate

test = sorted(generate("test"), key=lambda s: s.id)


def subset(n):
    groups = {}
    for s in test:
        groups.setdefault((s.tactic == Tactic.SPLIT_REQUEST, s.legit), []).append(s)
    rng = random.Random(0)
    return [s for g in groups.values() for s in rng.sample(g, min(len(g), n // 4))]


sim_client = CachedClient(OpenAICompatClient("http://127.0.0.1:11435/v1", "ollama"), OUT / "cache" / "sim")
summary = {}
for c in CONFIGS:
    scenarios = subset(c["subset"]) if c["subset"] else test
    api = RateLimitedClient(OpenAICompatClient(ENDPOINTS[c["provider"]], KEYS[c["provider"]], timeout_s=180), rpm=c["rpm"])
    agent_client = CachedClient(api, OUT / "cache" / c["model_id"], max_calls=6000)
    start = time.perf_counter()
    run_batch(
        [Job(s, r) for s in scenarios for r in range(1, c["runs"] + 1)],
        lambda s, r, c=c: LLMAgent(agent_client, c["model"], temperature=0.7, seed=run_seed(s, r, "agent") if c["seed"] else None),
        lambda s, r: LLMCustomer(sim_client, SIMULATOR, s, seed=run_seed(s, r, "customer"), max_tokens=250),
        OUT / f"{c['model_id']}.jsonl", model_id=c["model_id"], defense=Defense(c["defense"]), workers=4, log_every=40,
    )
    records = load_records(OUT / f"{c['model_id']}.jsonl")
    summary[c["model_id"]] = {"modelo": c["model"], "cenarios": len(scenarios), "conversas": len(records),
                              "horas": round((time.perf_counter() - start) / 3600, 2),
                              "vereditos": dict(Counter(r.verdict.value for r in records)),
                              "chamadas_ao_modelo": agent_client.calls_made}
    print(c["model_id"], json.dumps(summary[c["model_id"]], ensure_ascii=False))
shutil.rmtree(OUT / "cache", ignore_errors=True)
(OUT / "resumo.json").write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8")